In [ ]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Styling setup
sns.set_theme(context="talk", style="whitegrid")
pd.set_option("display.max_columns", 100)

# Resolve project root for modular imports
ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.eda import eda_summary

# Load processed data
data_path = ROOT_DIR / "data" / "processed" / "market_data_outliers_handled.parquet"
if not data_path.exists():
    data_path = ROOT_DIR / "data" / "raw" / "outliers_homework.csv"

df = pd.read_parquet(data_path) if data_path.suffix == ".parquet" else pd.read_csv(data_path)

# 1. Statistical Summary Output
summary_df = eda_summary(df)
display(summary_df)

# 2. Variable Distribution Plots (Histograms & Boxplots)
numeric_cols = df.select_dtypes(include=["float64", "int64"]).columns.tolist()
cols_to_plot = [c for c in numeric_cols if not c.startswith("outlier_")][:3]

fig, axes = plt.subplots(len(cols_to_plot), 2, figsize=(14, 4 * len(cols_to_plot)))
for i, col in enumerate(cols_to_plot):
    sns.histplot(df[col], kde=True, ax=axes[i, 0], color="steelblue")
    axes[i, 0].set_title(f"Distribution: {col}")
    
    sns.boxplot(x=df[col], ax=axes[i, 1], color="coral")
    axes[i, 1].set_title(f"Boxplot: {col}")

plt.tight_layout()
plt.show()

# 3. Bivariate Relationships & Correlation Matrix
if len(cols_to_plot) >= 2:
    plt.figure(figsize=(8, 5))
    sns.scatterplot(data=df, x=cols_to_plot[0], y=cols_to_plot[1], alpha=0.7, color="purple")
    plt.title(f"Bivariate Analysis: {cols_to_plot[0]} vs {cols_to_plot[1]}")
    plt.show()

    plt.figure(figsize=(8, 6))
    corr = df[cols_to_plot].corr()
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", vmin=-1, vmax=1)
    plt.title("Feature Correlation Matrix")
    plt.show()

# 4. Temporal Trend Visualization
if "date" in df.columns:
    df["date"] = pd.to_datetime(df["date"])
    df_time = df.sort_values("date").set_index("date")
    
    plt.figure(figsize=(12, 5))
    plt.plot(df_time.index, df_time[cols_to_plot[0]], label=cols_to_plot[0], alpha=0.6)
    plt.plot(df_time.index, df_time[cols_to_plot[0]].rolling(7, min_periods=1).mean(), label="7-Day Rolling Mean", color="red")
    plt.title("Time-Series Behavior & Trend Analysis")
    plt.xlabel("Date")
    plt.ylabel("Value")
    plt.legend()
    plt.show()